# GSM8K 深度–正确率评测：Qwen3-4B 的 thinking budget 实验

**模型**：`Qwen/Qwen3-4B` 原版（hybrid thinking，不是 `Qwen3-4B-Instruct-2507`）。
**数据**：GSM8K socratic 测试集 1319 题；深度 = gold solution 里 `<<...>>` 的个数（前面已经分好，本 notebook 不改）。
**条件**：5 个，每个条件单独写一个 jsonl。

| 条件 | thinking | vLLM `thinking_token_budget` |
|---|---|---|
| `thinking_off` | 关（`enable_thinking=False`） | — |
| `thinking_budget_256` | 开 | 256 |
| `thinking_budget_512` | 开 | 512 |
| `thinking_budget_1024` | 开 | 1024 |
| `thinking_unlimited` | 开 | 不设（受 `max_tokens` 限制） |

**产物目录**：`gsm8k_result/Qwen3-4B/`（相对 notebook 所在目录）

- 逐题记录：`thinking_off.jsonl`、`thinking_budget_256.jsonl`、`thinking_budget_512.jsonl`、`thinking_budget_1024.jsonl`、`thinking_unlimited.jsonl`
- 汇总：`summary_overall.csv`、`summary_by_depth.csv`、`accuracy_by_depth.png`

**运行**：只需要改第二格的 `MODEL_PATH`。thinking budget 依赖 `ReasoningConfig` + `SamplingParams(thinking_token_budget=...)`，需要较新的 vLLM。

In [ ]:
# ===== 全局配置：只需要改 MODEL_PATH =====
import os, re, json, time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

# 本地 Qwen3-4B 权重目录（Qwen/Qwen3-4B 原版），改成自己的路径
MODEL_PATH = "/path/to/Qwen3-4B"

SPLIT = "test"

# 采样参数沿用项目统一值；thinking 不限制时要留足输出长度
TEMPERATURE = 0.6
TOP_P = 0.95
SEED = 0
MAX_NEW_TOKENS = 8192
MAX_MODEL_LEN = 16384

# 结果全部写到 gsm8k_result/Qwen3-4B/，每个条件一个 jsonl
RESULTS_DIR = Path.cwd() / "gsm8k_result" / "Qwen3-4B"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# 数据目录：向上找到含 gsm8k_dataset 的仓库根
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "gsm8k_dataset").exists())
DATA_DIR = ROOT / "gsm8k_dataset"

# 五个条件：thinking 关闭 / thinking 打开 + budget 256、512、1024 / 不限制
CONDITIONS = [
    {"name": "thinking_off", "thinking": False, "budget": None},
    {"name": "thinking_budget_256", "thinking": True, "budget": 256},
    {"name": "thinking_budget_512", "thinking": True, "budget": 512},
    {"name": "thinking_budget_1024", "thinking": True, "budget": 1024},
    {"name": "thinking_unlimited", "thinking": True, "budget": None},
]

print("模型:", MODEL_PATH)
print("数据:", DATA_DIR)
print("结果:", RESULTS_DIR)
print("条件:", [c["name"] for c in CONDITIONS])

## 1. 数据与深度（保持不变）

深度直接由 gold answer 里的 `<<...>>` 数量得到，不依赖模型输出。

In [ ]:
# ===== 读取 GSM8K socratic 测试集，并按 <<...>> 计算深度 =====
ANN_RE = re.compile(r"<<[^<>]*>>")


def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def load_gsm8k(split="test"):
    local = DATA_DIR / f"{split}.jsonl"
    if local.exists():
        return pd.DataFrame(read_jsonl(local)), f"local socratic ({local.name})"
    from datasets import load_dataset
    ds = load_dataset("openai/gsm8k", "socratic", split=split)
    df = ds.to_pandas()
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    df.to_json(local, orient="records", lines=True, force_ascii=False)
    return df, "huggingface openai/gsm8k[socratic]"


raw_df, source = load_gsm8k(SPLIT)
df = raw_df.reset_index(drop=True)
df["uid"] = [f"{SPLIT}-{i:04d}" for i in range(len(df))]
df["depth"] = df["answer"].map(lambda s: len(ANN_RE.findall(str(s))))

print(f"数据来源: {source} | 题数: {len(df)}")
print(df[["question", "depth"]].head(3).to_string(index=False, max_colwidth=60))

dist = (df["depth"].value_counts().sort_index()
          .rename_axis("深度").reset_index(name="题数"))
dist["占比"] = dist["题数"] / len(df)
print("\n深度分布：")
display(dist.style.format({"占比": "{:.1%}"}))

## 2. Prompt 与答案解析

Prompt 直接走模型自带的 chat template：`enable_thinking=False` 时模板会预填一个空的 `<think></think>` 块，模型直接写答案；`True` 时模型自己生成 `<think> ... </think>`。

答案解析：gold 取 `####` 后的数字；预测优先取最后一个 `####`，其次 `\boxed{}`，最后取全文最后一个数字。

In [ ]:
# ===== Prompt 与答案抽取 =====
SYSTEM_PROMPT = """You are a math problem solver. Solve the problem carefully step by step.

The final line of your response MUST be exactly in this format:
#### <numeric answer>

where <numeric answer> is a single number (no units, no commas, no extra words). Do not write anything after the #### line."""

NUM_RE = re.compile(r"-?\d[\d,]*(?:\.\d+)?")


def build_messages(question):
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}]


def parse_number(text):
    """从一段文字里取最后一个数字；支持 $、逗号、Unicode 负号。"""
    if text is None:
        return None
    s = str(text).replace("−", "-").replace("$", "").replace("，", ",")
    nums = NUM_RE.findall(s)
    if not nums:
        return None
    try:
        return float(nums[-1].replace(",", ""))
    except ValueError:
        return None


def parse_gold(answer):
    """GSM8K gold：#### 后面的数字。"""
    s = str(answer)
    if "####" in s:
        return parse_number(s.rsplit("####", 1)[1])
    return parse_number(s.strip().splitlines()[-1])


def parse_pred(text):
    """模型输出：优先取最后一个 #### 行，其次 \boxed{}，最后取全文最后一个数。"""
    text = str(text).replace("<|im_end|>", "")
    if "####" in text:
        return parse_number(text.rsplit("####", 1)[1])
    m = re.search(r"\\boxed\{([^}]*)\}", text)
    if m:
        return parse_number(m.group(1))
    return parse_number(text)


def is_correct(pred, gold, tol=1e-4):
    return pred is not None and gold is not None and abs(pred - gold) <= tol


df["gold"] = df["answer"].map(parse_gold)
print("gold 解析失败:", int(df["gold"].isna().sum()))

## 3. vLLM 推理（thinking budget）

`LLM` 只加载一次，`reasoning_config` 告诉 vLLM 思考块从哪开始、到哪结束：

- 起点：`<|im_start|>assistant\n`（prompt 的结尾）
- 终点：`</think>\n\n`

每个条件一个 `SamplingParams`：`thinking_token_budget` 设成 256 / 512 / 1024，或者 `None`（不限制）。
预算用完时 vLLM 会强制模型输出 `</think>`，然后继续生成正式答案。

> 显存不够时调小 `MAX_MODEL_LEN` 或 `gpu_memory_utilization`；T4 等 Turing 卡把 `dtype` 改成 `"half"`。

In [ ]:
# ===== 用 vLLM 跑五个条件，每个条件一个 jsonl =====
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")  # 没有 nvcc 时避免 FlashInfer 采样器 JIT 失败

from vllm import LLM, SamplingParams
from vllm.config import ReasoningConfig

llm = LLM(
    model=MODEL_PATH,
    dtype="bfloat16",
    gpu_memory_utilization=0.85,
    max_model_len=MAX_MODEL_LEN,
    reasoning_config=ReasoningConfig(
        reasoning_start_str="<|im_start|>assistant\n",
        reasoning_end_str="</think>\n\n",
    ),
)
tokenizer = llm.get_tokenizer()

# 两种模式的 prompt 各准备一份，五个条件复用
prompts = {
    thinking: [tokenizer.apply_chat_template(
                   build_messages(q), tokenize=False,
                   add_generation_prompt=True, enable_thinking=thinking)
               for q in df["question"]]
    for thinking in (True, False)
}

records = df.to_dict("records")

for cond in CONDITIONS:
    name = cond["name"]
    path = RESULTS_DIR / f"{name}.jsonl"

    sp = SamplingParams(
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_tokens=MAX_NEW_TOKENS,
        seed=SEED,
        thinking_token_budget=cond["budget"],  # None = 不限制
    )

    t0 = time.time()
    outputs = llm.generate(prompts[cond["thinking"]], sp)
    elapsed = time.time() - t0

    rows = []
    for r, out in zip(records, outputs):
        text = out.outputs[0].text
        pred = parse_pred(text)
        rows.append({
            "condition": name,
            "thinking": cond["thinking"],
            "thinking_budget": cond["budget"],
            "uid": r["uid"],
            "depth": int(r["depth"]),
            "gold": r["gold"],
            "pred": pred,
            "correct": bool(is_correct(pred, r["gold"])),
            "n_tokens": len(out.outputs[0].token_ids),
            "finish_reason": out.outputs[0].finish_reason,
            "output": text,
        })

    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    acc = sum(row["correct"] for row in rows) / len(rows)
    avg_tok = sum(row["n_tokens"] for row in rows) / len(rows)
    print(f"{name}: acc={acc:.2%} | 平均 {avg_tok:.0f} tokens | 用时 {elapsed / 60:.1f} min -> {path}")

## 4. 汇总

In [ ]:
# ===== 总体正确率 + 按深度 =====
dfs = {c["name"]: pd.DataFrame(read_jsonl(RESULTS_DIR / f"{c['name']}.jsonl")) for c in CONDITIONS}

overall = pd.DataFrame([{
    "condition": name,
    "n": len(r),
    "correct": int(r["correct"].sum()),
    "acc": r["correct"].mean(),
    "avg_tokens": r["n_tokens"].mean(),
    "truncated": r["finish_reason"].eq("length").mean(),
} for name, r in dfs.items()])
print(overall.to_string(index=False))
overall.to_csv(RESULTS_DIR / "summary_overall.csv", index=False, encoding="utf-8-sig")

by_depth = pd.DataFrame({name: r.groupby("depth")["correct"].mean() for name, r in dfs.items()})
n_by_depth = next(iter(dfs.values())).groupby("depth").size().rename("题数")
print("\n按深度分组的正确率：")
print(pd.concat([n_by_depth, by_depth], axis=1).to_string())
by_depth.to_csv(RESULTS_DIR / "summary_by_depth.csv", encoding="utf-8-sig")

## 5. 正确率–深度图

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for name, r in dfs.items():
    g = r.groupby("depth")["correct"].mean()
    ax.plot(g.index, g.values, marker="o", label=name)
ax.set_xlabel("Depth (number of <<>> steps in gold solution)")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.02)
ax.set_xticks(sorted(n_by_depth.index))
ax.grid(alpha=0.25)
ax.legend(frameon=False)
ax.set_title(f"Qwen3-4B on GSM8K {SPLIT} (n={len(df)})")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "accuracy_by_depth.png", dpi=160, bbox_inches="tight")
plt.show()

## 6. 注意事项

1. **thinking 预算是硬限制**：到预算后 vLLM 强制输出 `</think>`，思考被截断，答案质量可能下降；这正是要测的东西。
2. **`thinking_unlimited` 不是真的无限**：仍受 `MAX_NEW_TOKENS=8192` 和 `MAX_MODEL_LEN=16384` 限制。`finish_reason == "length"` 比例高就调大这两个值（同时看显存）。
3. **单次采样**：`temperature=0.6` 沿项目统一值，重跑会有波动；要更稳可以改成 0 做贪心。
4. **base 模型指令遵循较弱**：没写 `####` 的样本会退化成「最后一个数字」，错误样例直接看 jsonl 里的 `output` 原文。